In [ ]:
import os
import sys
import numpy as np

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from utils import build_BBP_cell, compute_Ve_over_cell

from neuron import h
h.load_file('stdrun.hoc')
h.celsius = 34

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, TwoSlopeNorm
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
layer = 23
if layer == 23:
    model_name = 'L23_PC_cADpyr229_1'
elif layer == 5:
    model_name = 'L5_TTPC1_cADpyr232_1'
else:
    raise Exception('Only layers 2/3 and 5 are currently supported')
model_dir = os.path.join('..', 'BBP_models', model_name)
cell = build_BBP_cell(model_dir, add_synapses=0, replace_axon=0)

In [ ]:
Emag = 250 # [V/m]
θ    = -np.pi/2
ϕ    = np.pi/2
spherical_to_cartesian = lambda r,theta,phi: np.array([r*np.sin(theta)*np.cos(phi),
                                                       r*np.sin(theta)*np.sin(phi),
                                                       r*np.cos(theta)])
Efield = lambda X: spherical_to_cartesian(Emag,θ,ϕ)

In [ ]:
Ve,points = compute_Ve_over_cell(cell, Efield, full_output=True)

In [ ]:
data = np.vstack(list(points.values()))
X = data[:,:3]
V = data[:,-1]
norm = TwoSlopeNorm(vmin=V.min()*1e3, vcenter=0, vmax=V.max()*1e3)
fig,ax = plt.subplots(1, 1, figsize=(6,4))
plt.scatter(X[:,0], X[:,1], c=V*1e3, s=0.1, cmap='coolwarm', norm=norm)
plt.colorbar()
ax.axis('equal')
sns.despine()
fig.tight_layout()

In [ ]:
delay = 10
after = 10

stim_dur = 1 # [ms]
dt = min(h.dt, stim_dur/5)
after = 10
tstop = delay + stim_dur + after
time = np.r_[0 : tstop : dt]
stim = np.zeros_like(time)
idx = (time >= delay) & (time < delay+stim_dur)
stim[idx] = 1
t_vec = h.Vector(time)
E_vecs = []
for i,sec in enumerate(cell.all):
    sec.insert('extracellular')
    E_vecs.append([])
    for j,seg in enumerate(sec):
        vec = h.Vector(stim * Ve[i][j] * 1e3)
        vec.play(seg._ref_e_extracellular, t_vec, 1)
        E_vecs[i].append(vec)
h.dt = dt

In [ ]:
def add_recorders(rec, seclist, prefix):
    for i,sec in enumerate(seclist):
        for seg in sec:
            key = f'{prefix}[{i}]({seg.x:.3f})'
            rec[key] = h.Vector()
            rec[key].record(seg._ref_v)

rec = {'time': h.Vector(), 'soma(0.5)': h.Vector()}
rec['time'].record(h._ref_t)
rec['soma(0.5)'].record(cell.soma[0](0.5)._ref_v)

add_recorders(rec, cell.basal, 'basal')
add_recorders(rec, cell.apical, 'apical')
add_recorders(rec, cell.axon, 'axon')

In [ ]:
h.tstop = delay + stim_dur + after
h.cvode_active(0)
h.run()

In [ ]:
%matplotlib inline
fig,ax = plt.subplots(3, 1, figsize=(6,5), sharex=True, sharey=True)
t = np.array(rec['time']) - delay
idx = (t>=-2) & (t<10)
Vsoma = np.array(rec['soma(0.5)'])
cmap = plt.get_cmap('Accent')
i = 0
# for k,v in rec.items():
#     if 'axon' in k:
#         ax[0].plot(t[idx], np.array(v)[idx], color=cmap(i), lw=1.5, label=k)
#         i += 1
ax[0].plot(t[idx], Vsoma[idx], 'k', lw=2, label='soma')
ax[0].set_title('Stimulus: {:g} ms @ {:g} V/m'.format(stim_dur, Emag), fontsize=fontsize+1)
gray = 0.4 + np.zeros(3)
for i in range(23):
    ax[1].plot(t[idx], np.array(rec[f'basal[{i}](0.500)'])[idx], color=gray, lw=0.5)
for i in range(23):
    ax[2].plot(t[idx], np.array(rec[f'apical[{i}](0.500)'])[idx], color=gray, lw=0.5)

ax[1].set_title('Basal dendrites', fontsize=fontsize+1)
ax[2].set_title('Apical dendrites', fontsize=fontsize+1)
ax[-1].set_xlabel('Time (ms)')
xticks = np.r_[-2 : 11 : 2]
yticks = np.r_[-80 : 65 : 20]
for a  in ax:
    a.set_ylabel('Voltage (mV)')
    a.set_yticks(yticks)
    a.set_xticks(xticks)
    a.grid(which='major', ls=':', lw=0.5, color=[.6,.6,.6], axis='y')
ax[0].legend(loc='upper right', frameon=False, fontsize=8)
sns.despine()
fig.tight_layout()
plt.savefig('voltage_traces.pdf')